# MLP Medium Ensemble

In [1]:
import os, random
from itertools import product
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import joblib

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEEDS = [42, 123, 456, 789, 2026]
HIDDEN = [128, 64]

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

X_train=np.load("data/X_train_processed.npy")
X_val=np.load("data/X_val_processed.npy")
y_train=np.load("data/y_train_scaled.npy")
y_val=np.load("data/y_val_scaled.npy")
y_train_usd=np.load("data/y_train_usd.npy")
y_val_usd=np.load("data/y_val_usd.npy")
target_scaler=joblib.load("models/target_scaler_v2.joblib")
Y_MEAN=float(target_scaler["mean"]); Y_STD=float(target_scaler["std"])
print("Device:", DEVICE)
print("Train:", X_train.shape, "Val:", X_val.shape)

Device: cpu
Train: (934, 278) Val: (234, 278)


In [2]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden, dropout=0.0):
        super().__init__(); layers=[]; prev=input_dim
        for h in hidden:
            layers += [nn.Linear(prev,h), nn.ReLU()]
            if dropout>0: layers.append(nn.Dropout(dropout))
            prev=h
        layers.append(nn.Linear(prev,1)); self.net=nn.Sequential(*layers)
    def forward(self,x): return self.net(x).squeeze(1)

def scaled_to_usd(a): return np.asarray(a)*Y_STD+Y_MEAN

def train_model(hp, seed):
    set_seed(seed)
    model=MLP(X_train.shape[1],HIDDEN,hp["dropout"]).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=hp["lr"],weight_decay=hp["weight_decay"])
    loss_fn=nn.MSELoss()
    Xtr=torch.tensor(X_train,dtype=torch.float32); ytr=torch.tensor(y_train,dtype=torch.float32)
    Xva=torch.tensor(X_val,dtype=torch.float32).to(DEVICE)
    loader=DataLoader(TensorDataset(Xtr,ytr),batch_size=hp["batch_size"],shuffle=True, generator=torch.Generator().manual_seed(seed))
    best_rmse=float("inf"); best_state=None; best_epoch=-1; wait=0
    for epoch in range(hp["max_epochs"]):
        model.train()
        for xb,yb in loader:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE); opt.zero_grad()
            loss=loss_fn(model(xb),yb); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),5.0); opt.step()
        model.eval()
        with torch.no_grad(): vp=model(Xva).cpu().numpy()
        pred_usd=scaled_to_usd(vp)
        vr=float(np.sqrt(np.mean((pred_usd-y_val_usd)**2)))
        if vr < best_rmse-1e-6:
            best_rmse=vr; best_epoch=epoch; wait=0
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else: wait += 1
        if wait>=hp["patience"]: break
    model.load_state_dict(best_state); model.eval()
    return model,best_rmse,best_epoch

def predict_usd(model,X):
    with torch.no_grad(): p=model(torch.tensor(X,dtype=torch.float32).to(DEVICE)).cpu().numpy()
    return scaled_to_usd(p)

## Recuperar la mejor configuración Medium

In [3]:
grid={"lr":[3e-4,1e-3,2e-3], "dropout":[0.0,0.1,0.2], "weight_decay":[0.0,1e-4,1e-3]}
rows=[]; best_hp=None; best_rmse=float("inf")
for i,(lr,drop,wd) in enumerate(product(grid["lr"],grid["dropout"],grid["weight_decay"]),1):
    hp={"lr":lr,"dropout":drop,"weight_decay":wd,"batch_size":32,"max_epochs":600,"patience":50}
    _,rmse,epoch=train_model(hp,42)
    rows.append({**hp,"RMSE_USD":rmse,"best_epoch":epoch})
    print(f"[{i:02d}/27] lr={lr:.0e} drop={drop:.1f} wd={wd:.0e} -> ${rmse:,.2f}")
    if rmse<best_rmse: best_rmse=rmse; best_hp=hp.copy()
results_grid=pd.DataFrame(rows).sort_values("RMSE_USD").reset_index(drop=True)
display(results_grid.head())
print("HP seleccionados:",best_hp)
print(f"Mejor RMSE seed 42: ${best_rmse:,.2f}")

[01/27] lr=3e-04 drop=0.0 wd=0e+00 -> $27,170.23
[02/27] lr=3e-04 drop=0.0 wd=1e-04 -> $27,136.04
[03/27] lr=3e-04 drop=0.0 wd=1e-03 -> $27,132.48
[04/27] lr=3e-04 drop=0.1 wd=0e+00 -> $26,367.31
[05/27] lr=3e-04 drop=0.1 wd=1e-04 -> $26,402.87
[06/27] lr=3e-04 drop=0.1 wd=1e-03 -> $26,274.49
[07/27] lr=3e-04 drop=0.2 wd=0e+00 -> $27,243.41
[08/27] lr=3e-04 drop=0.2 wd=1e-04 -> $27,214.41
[09/27] lr=3e-04 drop=0.2 wd=1e-03 -> $27,201.78
[10/27] lr=1e-03 drop=0.0 wd=0e+00 -> $26,330.43
[11/27] lr=1e-03 drop=0.0 wd=1e-04 -> $26,299.32
[12/27] lr=1e-03 drop=0.0 wd=1e-03 -> $26,383.27
[13/27] lr=1e-03 drop=0.1 wd=0e+00 -> $25,958.87
[14/27] lr=1e-03 drop=0.1 wd=1e-04 -> $26,184.93
[15/27] lr=1e-03 drop=0.1 wd=1e-03 -> $25,913.65
[16/27] lr=1e-03 drop=0.2 wd=0e+00 -> $26,459.70
[17/27] lr=1e-03 drop=0.2 wd=1e-04 -> $25,982.32
[18/27] lr=1e-03 drop=0.2 wd=1e-03 -> $26,414.26
[19/27] lr=2e-03 drop=0.0 wd=0e+00 -> $25,341.09
[20/27] lr=2e-03 drop=0.0 wd=1e-04 -> $26,904.06
[21/27] lr=2e-03 dro

,lr,dropout,weight_decay,batch_size,max_epochs,patience,RMSE_USD,best_epoch
0,0.002,0.2,0.0001,32,600,50,23768.275391,33
1,0.002,0.2,0.0000,32,600,50,24123.939453,45
2,0.002,0.1,0.0001,32,600,50,24638.437500,72
3,0.002,0.1,0.0010,32,600,50,24654.048828,10
4,0.002,0.2,0.0010,32,600,50,25045.257812,20


HP seleccionados: {'lr': 0.002, 'dropout': 0.2, 'weight_decay': 0.0001, 'batch_size': 32, 'max_epochs': 600, 'patience': 50}
Mejor RMSE seed 42: $23,768.28


## Entrenar 5 Medium con semillas distintas

In [4]:
os.makedirs("models/medium_ensemble",exist_ok=True); os.makedirs("reports",exist_ok=True)
models=[]; val_preds=[]; seed_rows=[]
for seed in SEEDS:
    model,rmse,epoch=train_model(best_hp,seed)
    pred=predict_usd(model,X_val)
    models.append(model); val_preds.append(pred)
    seed_rows.append({"seed":seed,"RMSE_USD":rmse,"best_epoch":epoch})
    torch.save({"model_state_dict":model.state_dict(),"architecture_name":"medium","hidden_layers":HIDDEN,"hyperparams":best_hp,"seed":seed,"val_rmse_usd":rmse,"target_mean":Y_MEAN,"target_std":Y_STD}, f"models/medium_ensemble/medium_seed_{seed}.pt")
    print(f"Seed {seed}: RMSE val=${rmse:,.2f} | epoch={epoch}")
seed_df=pd.DataFrame(seed_rows).sort_values("RMSE_USD")
display(seed_df)

Seed 42: RMSE val=$23,768.28 | epoch=33
Seed 123: RMSE val=$24,142.68 | epoch=69
Seed 456: RMSE val=$23,171.97 | epoch=45
Seed 789: RMSE val=$25,921.70 | epoch=68
Seed 2026: RMSE val=$24,262.57 | epoch=48


,seed,RMSE_USD,best_epoch
2,456,23171.968750,45
0,42,23768.275391,33
1,123,24142.681641,69
4,2026,24262.572266,48
3,789,25921.699219,68


## Evaluar el ensemble en validación

In [5]:
ensemble_val=np.mean(np.stack(val_preds,axis=0),axis=0)
ensemble_rmse=float(np.sqrt(np.mean((ensemble_val-y_val_usd)**2)))
print(f"RMSE Ensemble Validación: ${ensemble_rmse:,.2f}")
print(f"Mejor modelo individual:  ${seed_df.RMSE_USD.min():,.2f}")
print("Objetivo < $30,000:", "ALCANZADO" if ensemble_rmse<30000 else "NO")
results_grid.to_csv("reports/medium_ensemble_grid_original.csv",index=False)
seed_df.to_csv("reports/medium_ensemble_seeds.csv",index=False)
joblib.dump({"seeds":SEEDS,"hidden_layers":HIDDEN,"hyperparams":best_hp,"ensemble_val_rmse":ensemble_rmse},"models/medium_ensemble/config.joblib")
print("Guardados 5 checkpoints en models/medium_ensemble/")

RMSE Ensemble Validación: $23,133.46
Mejor modelo individual:  $23,171.97
Objetivo < $30,000: ALCANZADO
Guardados 5 checkpoints en models/medium_ensemble/
